# Artificial Intelligence in Practice — Lab 1
# 📧 Pragmatics & Intent Extraction with LLMs

@ Instructor: BME TMIT

In this laboratory session, we will work with a sample from the **Enron email corpus**. Our goal is to demonstrate how to use generative AI to automate and transform a complex linguistic task—extracting the **pragmatic intent (Speech Act)** behind emails—into structured JSON data, a task that is notoriously difficult for traditional algorithmic methods.

### 💡 Lab Structure:
Instead of passive tutorials, this lab is built around **Challenges**. You will evaluate traditional baselines, design strict JSON-enforced prompts, handle linguistic edge cases, and cross-evaluate models on unlabeled data.

---
## 0. Setup ⚙️

Preparing the infrastructure: installing packages and loading the open-weights **Qwen2.5-1.5B-Instruct** language model onto the Colab T4 GPU.

### 🖥️ Checking GPU enablement
Make sure your Colab runtime is set to use a GPU:
**Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save**

In [ ]:
# Installing necessary packages
!pip install -q transformers accelerate pandas tqdm
print('✅ Packages installed.')

In [ ]:
import pandas as pd
import numpy as np
import re, json, time, io
import torch
from transformers import pipeline

# Loading an open-weights LLM (runs locally on the Colab GPU, no API key required)
LLM_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
print(f'⏳ Loading {LLM_MODEL} to the GPU…')

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
_gen = pipeline('text-generation', model=LLM_MODEL, torch_dtype=dtype, device_map='auto')

def ask_llm(prompt, as_json=False):
    messages = [{'role': 'user', 'content': prompt}]
    out = _gen(messages, max_new_tokens=512, do_sample=False, pad_token_id=_gen.tokenizer.eos_token_id)
    text = out[0]['generated_text'][-1]['content'].strip()
    if as_json:
        m = re.search(r'[\[{].*[\]}]', text, re.DOTALL)
        return m.group(0) if m else text
    return text

device = 'GPU ✅' if torch.cuda.is_available() else 'CPU (Slower) ⚠️'
print(f'✅ Model loaded. Runtime environment: {device}')

---
## 1. Loading and Preparing Data 📥

### 📖 Theory: What is a Speech Act and Intent Extraction?
In linguistics, **pragmatics** doesn't examine what words *mean* (semantics), but rather what we *achieve* with them in a given context (**Speech Act Theory**).
- **Directive (Action Required):** An instruction or request (e.g., *"Send me the report!"*).
- **Commissive (Commitment):** A promise or obligation (e.g., *"I'll do it by tomorrow."*).
- **Representative (FYI Only):** Purely sharing information (e.g., *"The agenda is attached."*).

To ensure the lab runs efficiently, we will take a random sample of 50 emails from the uploaded dataset.

In [ ]:
# Uploading the raw dataset and sampling
try:
    from google.colab import files
    print("Please upload your Enron CSV file.\nExpected columns: message_id, date, from, to, cc, subject, body")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    raw_df = pd.read_csv(io.BytesIO(uploaded[filename]))
    print(f"\n✅ Dataset loaded successfully: {len(raw_df)} rows.")
    
    SAMPLE_SIZE = 50
    df = raw_df.sample(n=min(SAMPLE_SIZE, len(raw_df)), random_state=42).reset_index(drop=True)
    print(f"Working with a sample of {len(df)} rows (out of {len(raw_df)} uploaded).")
    
except Exception as e:
    print("⚠️ File upload skipped or failed. Ensure you are running this in Google Colab.")
    print(f"Error details: {e}")
    df = pd.DataFrame() # Empty fallback

if not df.empty:
    display(df.head(4))

### 🎯 Challenge 1: Data Exploration
Write code that displays:
1. The top 5 most frequent senders (`from`).
2. The number of missing (NaN) values in the `body` column.

In [ ]:
# Write your code here!

---
## 2. The Traditional Baseline (Rule-based / Regex) 🦖

### 📖 Theory: Why does keyword matching fail with pragmatics?
In traditional software engineering, we use regular expressions (Regex) to search for words (e.g., *"ASAP"*, *"please"*, *"need"*). However, due to pragmatics, the presence of these words **does not equal** intent:
- *Example A:* "I **need** the report by 5 PM." -> **Action Required** (A true task)
- *Example B:* "I got the report you **need**ed." -> **FYI Only** (Purely informational, yet contains the word 'need'!)

### 🎯 Challenge 2: Evaluating the Baseline
Below is a naive rule-based classifier. Apply it to the `df['body']` column and save the results into a new `regex_label` column! Print the distribution of the categories. Then, inspect a few results and consider why this approach is fundamentally flawed.

In [ ]:
def regex_classifier(text):
    text_lower = str(text).lower() # Casting to string to handle NaNs in raw data
    if re.search(r'\b(please|asap|need|can you|review|fix)\b', text_lower):
        return 'Action_Required'
    elif re.search(r'\b(will|promise|shall|submit|handle)\b', text_lower):
        return 'Commitment'
    else:
        return 'FYI_Only'

# Apply the function to the DataFrame here:

---
## 3. Generative AI — Zero-Shot Intent Classification ✨

### 📖 Theory: Zero-shot classification and JSON Schema enforcement
Based on their internal parametric knowledge, LLMs understand the context of the sentence and the underlying speech act. They don't search for keywords; they **interpret intent**. To integrate the LLM's output into automated software systems, we must force the response into a **strictly structured JSON format**.

▶️ **Demo: Talking to the LLM (API Basics)**
This is a minimal example of how to send a prompt to the model and force a JSON response. **This is not the solution to the challenge below**, but it shows you the mechanics of the `ask_llm` function.

In [ ]:
# DEMO: Basic LLM interaction with JSON enforcement
demo_email = str(df['body'].iloc[3]) if not df.empty else 'I got the report you needed.'

prompt = (
    "You are an AI assistant.\n"
    "Classify the intent of the following email as Action_Required, Commitment, or FYI_Only.\n\n"
    f"EMAIL: {demo_email}\n\n"
    "Return ONLY JSON: {\"intent\": \"...\"}"
)

print('Email:', demo_email[:100], '...')
print('LLM Response:', ask_llm(prompt, as_json=True))

### 🎯 Challenge 3: Zero-Shot Prompting and Batching
Your task is to replace the Regex with the LLM.
1. Write a zero-shot prompt instructing the model to classify emails into `Action_Required`, `Commitment`, or `FYI_Only`.
2. Enforce a strict JSON array output format in your prompt instructions.
3. Write a loop that processes the dataset in **batches of 5 emails** (to speed up inference). For each batch, pass the combined emails into a single prompt.
4. Parse the JSON response, extract the intents, and save them to a new column called `zero_shot_label`.

In [ ]:
# Write your batching and LLM prompt code here!

---
## 4. Multi-Labeling & Edge Cases Handling (Few-Shot Scaffolding) 🎓

### 📖 Theory: What if the email falls into multiple categories or contains a complaint?
In reality, emails are complex:
1. **Multi-Labeling:** An email can simultaneously be a *Complaint* and an *Action_Required*.
2. **Uncertainty / Other:** If the model receives an email that doesn't fit the 3 fixed categories, forcing a single label will cause errors.

**AI Engineering Solution:**
- **JSON Array Output:** Returning multiple labels in an array (`"intents": ["Complaint", "Action_Required"]`).
- **Catch-all Category:** Supporting an `Other` or `Complaint` category with a custom description.
- **Few-Shot Examples:** Providing explicit examples in the prompt to anchor ambiguous cases.

### 🎯 Challenge 4: Designing a Few-Shot Multi-Label Prompt
Design a new prompt that supports multiple labels per email and introduces a `Complaint` category. Include 2-3 explicit examples (few-shot) in the prompt to show the model exactly how to format the JSON array and handle complex intents. Run this over your dataset and save the joined results (e.g., `'Action_Required,Complaint'`) into a `few_shot_label` column.

In [ ]:
# Write your few-shot prompt and execution code here!

---
## 5. Model Cross-Evaluation & Human-in-the-Loop 🧑‍⚖️

### 📖 Theory: Unsupervised Evaluation
Since our raw dataset does not have a pre-labelled `ground_truth` created by an expert, we cannot simply calculate an "accuracy score". 
Instead, in real-world AI engineering, we do **Model Cross-Evaluation**: we compare the outputs of our Baseline (Regex) against our advanced LLM (Few-Shot). By investigating where they **disagree**, we can quickly spot the flaws of the traditional method.

### 🎯 Challenge 5: Error Analysis
Let's find the pragmatic gap! Write code to filter the DataFrame for rows where the **Regex predicted `FYI_Only`**, but the **Few-Shot LLM found `Action_Required` or `Complaint`**. 

*(Hint: Remember that the Few-Shot model outputs multiple labels as a comma-separated string, so use string containment or set comparison!)*

Display the email body and the labels. Read a few examples—who was right?

In [ ]:
# Write your evaluation code here!

---
## 6. Saving Results 💾

Finally, save your structured outputs into a CSV file.

In [ ]:
# Write your CSV export code here